# Kaggle Prerequisites: One-Time Setup

This notebook guides you through the one-time setup required before running the training pipeline on Kaggle.

**Run this locally (or in any terminal with Kaggle API access) — not inside a Kaggle notebook.**

## What you'll do
1. Download Common Voice Swahili (20GB) from Mozilla
2. Upload it to Kaggle as a reusable Dataset (`cv-swahili`)
3. Upload the repo code as a Kaggle Dataset (`hibiki-sw-code`)
4. Verify everything is accessible inside Kaggle

**English audio:** The En->Sw pipeline direction uses FLEURS English, which is downloaded automatically from HuggingFace at runtime — no manual download needed.

## Why Kaggle Datasets?
Kaggle notebooks reset every session. By storing the data as Kaggle Datasets,
you attach them at notebook start and they're available instantly at `/kaggle/input/` —
no re-downloading, no waiting.

## Full Pipeline Overview
```
[00a] One-time setup  (this notebook, run locally)
  -> Upload cv-swahili, hibiki-sw-code to Kaggle Datasets

[00b] Data pipeline   (run on Kaggle, twice)
  Run 1: sw->en  (uses cv-swahili + pretrained mms-tts-eng)
  Run 2: en->sw  (uses FLEURS English from HuggingFace + pretrained mms-tts-swh)
  -> Save output as Kaggle Dataset: hibiki-sw-pipeline-out

[00_data_preparation] Tokenizer + Mimi encoding  (run on Kaggle)
  -> Attach: cv-swahili, hibiki-sw-code, hibiki-sw-pipeline-out
  -> Save output as Kaggle Dataset: hibiki-sw-data

[Stage 1-4 training]  (run on Kaggle)
  -> Attach: hibiki-sw-data, hibiki-sw-code
```

In [ ]:
# Install Kaggle API (run locally)
# pip install kaggle

# Configure Kaggle credentials:
# 1. Go to https://www.kaggle.com/settings -> API -> Create New Token
# 2. Save the downloaded kaggle.json to ~/.kaggle/kaggle.json
# 3. chmod 600 ~/.kaggle/kaggle.json  (Linux/Mac)

import os
kaggle_json = os.path.expanduser('~/.kaggle/kaggle.json')
if os.path.exists(kaggle_json):
    print('Kaggle credentials found.')
else:
    print('ERROR: ~/.kaggle/kaggle.json not found.')
    print('Download it from https://www.kaggle.com/settings -> API -> Create New Token')

## Step 1: Download Common Voice Swahili

Get your credentials from:
https://commonvoice.mozilla.org/en/datasets

1. Select **Common Voice Corpus 19.0**
2. Select language: **Swahili (sw)**
3. Enter your email and download — you'll get a direct download link or use the Data Collective API

Alternatively use the Mozilla Data Collective API (requires registration):
https://datacollective.mozillafoundation.org/datasets/cmn3ailbd008nmb07mjyu3xro

In [ ]:
import os

CLIENT_ID = 'YOUR_CLIENT_ID_HERE'    # CHANGE THIS
ACCESS_KEY = 'YOUR_ACCESS_KEY_HERE'  # CHANGE THIS

print('=== Run these commands in your terminal ===')
print()
print('# Download Swahili (~20GB):')
print(f'curl -L -o ~/cv-sw.tar.gz \\')
print(f'    -H "X-Client-Id: {CLIENT_ID}" \\')
print(f'    -H "X-Access-Key: {ACCESS_KEY}" \\')
print(f'    "https://datacollective.mozillafoundation.org/api/v1/datasets/cmn3ailbd008nmb07mjyu3xro/download"')
print()
print('# Extract:')
print('mkdir -p ~/cv-corpus && tar -xzf ~/cv-sw.tar.gz -C ~/cv-corpus')
print()
print('# Verify:')
print('ls ~/cv-corpus/')

## Step 2: Upload Validated Common Voice Swahili to Kaggle

We only upload clips referenced in `validated.tsv` — this skips unvalidated audio and significantly reduces upload size (typically ~5-8GB instead of 20GB).

In [ ]:
import json, os, shutil, csv

KAGGLE_USERNAME = 'YOUR_KAGGLE_USERNAME'  # CHANGE THIS

CV_CORPUS_DIR = os.path.expanduser('~/cv-corpus')
sw_dir = f'{CV_CORPUS_DIR}/cv-corpus-19.0-2024-09-13/sw'

# Output directory containing only validated clips
validated_upload_dir = os.path.expanduser('~/cv-swahili-validated')
validated_clips_dir = f'{validated_upload_dir}/clips'
os.makedirs(validated_clips_dir, exist_ok=True)

# Read validated.tsv and collect clip filenames
validated_tsv = f'{sw_dir}/validated.tsv'
clip_filenames = set()
with open(validated_tsv, newline='', encoding='utf-8') as f:
    reader = csv.DictReader(f, delimiter='\t')
    for row in reader:
        clip_filenames.add(row['path'])

print(f'Validated clips referenced in validated.tsv: {len(clip_filenames)}')

# Copy only validated clips
src_clips_dir = f'{sw_dir}/clips'
copied = 0
missing = 0
for filename in clip_filenames:
    src = os.path.join(src_clips_dir, filename)
    dst = os.path.join(validated_clips_dir, filename)
    if os.path.exists(src):
        shutil.copy2(src, dst)
        copied += 1
    else:
        missing += 1

print(f'Copied: {copied}, Missing: {missing}')

# Copy TSV files (validated + splits)
for tsv in ['validated.tsv', 'train.tsv', 'dev.tsv', 'test.tsv']:
    src = os.path.join(sw_dir, tsv)
    if os.path.exists(src):
        shutil.copy2(src, validated_upload_dir)
        print(f'Copied {tsv}')

# Write Kaggle dataset metadata
sw_metadata = {
    'title': 'cv-swahili',
    'id': f'{KAGGLE_USERNAME}/cv-swahili',
    'licenses': [{'name': 'CC-BY-4.0'}]
}
with open(f'{validated_upload_dir}/dataset-metadata.json', 'w') as f:
    json.dump(sw_metadata, f, indent=2)

upload_size_gb = sum(
    os.path.getsize(os.path.join(dp, f))
    for dp, _, files in os.walk(validated_upload_dir)
    for f in files
) / (1024 ** 3)
print(f'\nUpload directory size: {upload_size_gb:.1f} GB')
print(f'Ready to upload: {validated_upload_dir}')
print()
print('=== Run this in your terminal to upload ===')
print(f'kaggle datasets create -p {validated_upload_dir} --dir-mode tar')

## Step 3: Upload Repo as Kaggle Dataset

The notebooks import from the repo at runtime. Upload it once as a Kaggle Dataset.

In [ ]:
import json, os, subprocess

KAGGLE_USERNAME = 'YOUR_KAGGLE_USERNAME'  # CHANGE THIS (same as above)

# Path to the hibiki-sw directory in your local clone
REPO_ROOT = os.path.abspath(os.path.join(os.path.dirname('__file__'), '..'))
print(f'Repo root: {REPO_ROOT}')

repo_metadata = {
    'title': 'hibiki-sw-code',
    'id': f'{KAGGLE_USERNAME}/hibiki-sw-code',
    'licenses': [{'name': 'other'}]
}
with open(f'{REPO_ROOT}/dataset-metadata.json', 'w') as f:
    json.dump(repo_metadata, f, indent=2)

print()
print('=== Upload repo to Kaggle ===')
print(f'kaggle datasets create -p {REPO_ROOT}')
print()
print('To update the repo after code changes:')
print(f'kaggle datasets version -p {REPO_ROOT} -m "Update code"')

## Step 4: Verify Inside Kaggle

Run this cell inside a Kaggle notebook (after attaching the datasets) to confirm everything is accessible.

In [ ]:
# Run this cell INSIDE a Kaggle notebook to verify all datasets are attached correctly
import os

checks = {
    'cv-swahili clips':    '/kaggle/input/cv-swahili/clips',
    'cv-swahili TSV':      '/kaggle/input/cv-swahili/validated.tsv',
    'hibiki-sw-code repo': '/kaggle/input/hibiki-sw-code/hibiki-sw/data',
}

all_ok = True
for name, path in checks.items():
    exists = os.path.exists(path)
    status = '[OK]' if exists else '[MISSING]'
    if not exists:
        all_ok = False
    print(f'{status} {name}: {path}')

print()
if all_ok:
    n_clips = len(os.listdir('/kaggle/input/cv-swahili/clips'))
    print(f'Clips available: {n_clips}')
    print('All datasets verified. Ready to run 00b_data_pipeline.ipynb')
    print('NOTE: En->Sw direction will use FLEURS English (auto-downloaded from HuggingFace — no dataset needed).')
else:
    print('Some datasets are missing. Attach them via Notebook Settings -> Add Data')
    print('Dataset names to search for: cv-swahili, hibiki-sw-code')